In [1]:
%run ../../../imports_common.py -e "encoding-stf"

In [ ]:
import  numpy  as np
import  pandas as pd
import  plotly.express         as px
import  plotly.graph_objects   as go
from    enum                   import StrEnum, auto
from    typing                 import Any, Dict, List, Optional, Union
from    IPython.display        import display
from    sentence_transformers  import SentenceTransformer
# package modules
from    velari_core.core       import Experiment, read_cache_dir
from    velari_ai.integrations.sklearn.clustering import HDBSCANDriver, KMeansDriver
from    velari_ai.integrations.sklearn.projection import ProjectionMode, UMAPProjection
from    velari_ai.integrations.sklearn.scoring    import scorer_clustering

exp: Experiment    # created by `%run imports_common.py` above

# Encoding text with `sentence-transformers`

Encode text directly with `sentence_transformers` (no gateway) and attach the vectors to a DataFrame.

**Choosing a model** — pick by language, task, and size/speed. Vectors are `float` lists of the listed dimension:

| Model | Dim | Max tokens | Params | Latency (ms/text) | Best for |
|---|---|---|---|---|---|
| `all-MiniLM-L6-v2` | 384 | 256 | ~23M | 0.18 | smaller and faster; good general-purpose default for prototyping |
| `all-MiniLM-L12-v2` | 384 | 128 | ~33M | 0.35 | deeper MiniLM variant; general-purpose, but a shorter input limit |
| `all-mpnet-base-v2` | 768 | 384 | ~109M | 0.56 | better general-purpose quality; slower, larger vectors |
| `multi-qa-MiniLM-L6-cos-v1` | 384 | 512 | ~23M | 0.21 | semantic search: short questions matched to longer passages |
| `paraphrase-multilingual-MiniLM-L12-v2` | 384 | 128 | ~118M | 0.31 | multilingual text (50+ languages); paraphrase/similarity |
| `BAAI/bge-small-en-v1.5` | 384 | 512 | ~33M | 0.34 | English retrieval; small and fast |
| `BAAI/bge-base-en-v1.5` | 768 | 512 | ~109M | 0.63 | English retrieval; stronger, slower |
| `BAAI/bge-m3` | 1024 | 8192 | ~568M | 1.77 | long documents and multilingual retrieval; much larger and slower |
| `google/embeddinggemma-300m` | 768 | 2048 | ~308M | — | multilingual (100+ languages) retrieval and similarity; Matryoshka vectors truncate to 512/256/128 dims; full-precision baseline; gated on Hugging Face (accept the license) |
| `google/embeddinggemma-300m-qat-q4_0-unquantized` | 768 | 2048 | ~308M | — | same model trained for 4-bit quantization (QAT); these weights are unquantized, so quantize to q4_0 downstream for the smallest footprint; gated |
| `google/embeddinggemma-300m-qat-q8_0-unquantized` | 768 | 2048 | ~308M | — | same, trained for 8-bit quantization (QAT); closer to full-precision quality; gated |
| `google/embeddinggemma-2` | 768 | 8192 | ~740M | — | successor: text, images, video and audio in one shared space (Matryoshka 128–768 dims); the 270M text backbone loads on its own for text-only use; Apache-2.0, not gated |

*Latency* is the median per-text encode time on an Apple-silicon GPU (MPS): 64 short ticket sentences, batch size 32, median of 5 runs after a warm-up. Treat it as relative — it varies with hardware, batch size, and text length. Inputs longer than *Max tokens* are truncated. Vectors from different models aren't comparable — index and query with the same one.

**Choosing the method by objective** — `encode_texts` (below) picks the `SentenceTransformer` method from an `Objective`:

| Objective | Method | Use it for |
|---|---|---|
| `SYMMETRIC` (default) | `encode` | comparing texts to each other: similarity, clustering, deduplication |
| `QUERY` | `encode_query` | the search question in retrieval |
| `DOCUMENT` | `encode_document` | the corpus records that questions are matched against |

`QUERY` and `DOCUMENT` apply the model's own `query` / `document` prompts (listed in `model.prompts`) and, for routed models, the matching task. The embeddinggemma models define those prompts, so the objective changes their vectors. The MiniLM, `bge`, and `multi-qa` models in the table define none, so all three objectives return the same vectors.

**Switching models** — set `MODEL_NAME` in the model cell and restart the kernel; nothing else changes. `google/embeddinggemma-2` needs `sentence-transformers[image]>=6.1` and `transformers>=5.19`, which the workspace now requires.

In [3]:
MODEL_NAME = "google/embeddinggemma-2"
#MODEL_NAME = "all-MiniLM-L6-v2"    # swap for any model in the table above
model      = SentenceTransformer(MODEL_NAME, cache_folder=read_cache_dir(app="huggingface/hub"))
enc: Dict[str, Any] = dict(normalize_embeddings=True, batch_size=32, show_progress_bar=False)
#model.max_seq_length

Loading weights:   0%|          | 0/1376 [00:00<?, ?it/s]

## Dataset

Every example below uses one tabular dataset, the customer-support FAQ (`data/cs/datasets/faq.csv`): 20 rows with `id`, `question`, `answer`, and `category` (8 categories). The questions are the texts to encode, and `category` labels the plots.

In [4]:
faq_path = exp.data_dir / "cs/datasets/faq.csv"    # noqa: F821 -- `exp` comes from imports_common.py
df_faq   = pd.read_csv(faq_path)
display(df_faq[["question", "category"]].head())
print(df_faq.shape, df_faq["category"].nunique())

,question,category
0,How do I reset my password?,Account
1,What's your refund policy?,Billing
2,How do I cancel my subscription?,Billing
3,What payment methods do you accept?,Billing
4,How do I update my profile information?,Account


(20, 4) 8


In [5]:
def to_frame(
    data:     Union[List[str], np.ndarray, pd.DataFrame],
    vectors:  np.ndarray,
    text_col: str = "text",
    emb_col:  str = "emb",
) -> pd.DataFrame:
    """Attach encoded vectors to their texts (list, array, or DataFrame) as a column of lists.

    Args:
        data (Union[List[str], np.ndarray, pd.DataFrame]): Encoded texts; a DataFrame keeps its columns and isn't mutated.
        vectors (np.ndarray): `model.encode()` output, one row per text.
        text_col (str): Text column name when `data` isn't a DataFrame.
        emb_col (str): Column that receives one `List[float]` per record.

    Returns:
        pd.DataFrame: `data` (or a new text column) plus `emb_col`.

    Examples:
        >>> df_emb = to_frame(tickets, model.encode(tickets, normalize_embeddings=True))
    """
    df_base = data if isinstance(data, pd.DataFrame) else pd.DataFrame({text_col: [str(t) for t in data]})
    return df_base.assign(**{emb_col: vectors.tolist()})

In [6]:
class Objective(StrEnum):
    """What the encoded text is for; selects the `SentenceTransformer` method (and its prompt)."""
    SYMMETRIC  = auto()   # text-to-text similarity or clustering -> model.encode
    QUERY      = auto()   # a search question                     -> model.encode_query
    DOCUMENT   = auto()   # a corpus record to be retrieved       -> model.encode_document


def encode_texts(
    model:     SentenceTransformer,
    texts:     List[str],
    objective: Objective = Objective.SYMMETRIC,
    **kwargs:  Any,
) -> np.ndarray:
    """Encode texts with the `SentenceTransformer` method that matches the objective.

    Args:
        model (SentenceTransformer): Loaded model; its `model.prompts` decide whether QUERY/DOCUMENT differ from SYMMETRIC.
        texts (List[str]): Texts to encode.
        objective (Objective): SYMMETRIC -> `encode`, QUERY -> `encode_query`, DOCUMENT -> `encode_document`.
        **kwargs (Any): Passed through to the chosen method, e.g. `normalize_embeddings=True`.

    Returns:
        np.ndarray: One row per text.

    Examples:
        >>> model = SentenceTransformer("google/embeddinggemma-2")
        >>> doc_vecs  = encode_texts(model, ["Invoice ACC-10293 was charged twice this month"], Objective.DOCUMENT)
        >>> query_vec = encode_texts(model, ["Why was invoice ACC-10293 charged twice?"], Objective.QUERY)[0]
    """
    methods = {
        Objective.SYMMETRIC: model.encode,
        Objective.QUERY:     model.encode_query,
        Objective.DOCUMENT:  model.encode_document,
    }
    return methods[objective](texts, **kwargs)

## List or array input

In [7]:
questions  = df_faq["question"].tolist()
vectors    = encode_texts(model, questions, **enc)    # ndarray (n, dim); SYMMETRIC -> model.encode; np.array(questions) works too
list_emb   = vectors.tolist()                         # plain list of vectors
df_emb     = to_frame(questions, vectors)             # DataFrame: text, emb
df_arr_emb = to_frame(np.array(questions), vectors, text_col="question")
display(df_emb.head())
print(vectors.shape, type(list_emb), type(df_emb["emb"][0]))

,text,emb
0,How do I reset my password?,"[-0.0181884765625, 0.02734375, 0.05908203125, ..."
1,What's your refund policy?,"[-0.033935546875, 0.02392578125, 0.03540039062..."
2,How do I cancel my subscription?,"[-0.0203857421875, -0.00927734375, 0.030395507..."
3,What payment methods do you accept?,"[-0.033203125, 0.031982421875, 0.06005859375, ..."
4,How do I update my profile information?,"[0.0264892578125, 0.000919342041015625, 0.0273..."


(20, 768) <class 'list'> <class 'list'>


## DataFrame input

In [8]:
doc_vecs = encode_texts(model, df_faq["question"].tolist(), Objective.DOCUMENT, **enc)    # the FAQ questions are the corpus to retrieve
df_emb   = to_frame(df_faq, doc_vecs)                                                     # default "emb"
df_emb   = to_frame(df_emb, doc_vecs, emb_col="text_emb")                                 # custom name
print(df_emb.columns.tolist(), "emb" in df_faq.columns)

['id', 'question', 'answer', 'category', 'emb', 'text_emb'] False


In [9]:
display(df_faq.head(1))
display(df_emb.head(1))

,id,question,answer,category
0,8226032e-dbb6-4337-9352-9cb18b8bca47,How do I reset my password?,Go to Settings > Security > Reset Password. Yo...,Account


,id,question,answer,category,emb,text_emb
0,8226032e-dbb6-4337-9352-9cb18b8bca47,How do I reset my password?,Go to Settings > Security > Reset Password. Yo...,Account,"[-0.014404296875, 0.0174560546875, 0.045654296...","[-0.014404296875, 0.0174560546875, 0.045654296..."


## Preprocessing

`model.preprocess` runs the tokenization step of `encode` (truncating to `model.max_seq_length`) and returns tensors: `input_ids`, `attention_mask`, and so on. It takes a list of texts, so convert an array with `.tolist()`. Summing `attention_mask` gives the token count per text, which shows what gets truncated before you encode. It doesn't include a query/document prompt, so on a prompted model the real count is a few tokens higher.

In [10]:
texts: List[Any] = df_faq["answer"].head(4).tolist() + [" ".join(df_faq["answer"])]    # a few answers, plus every answer joined into one long text
features    = model.preprocess(texts)
df_tokens   = (
    pd.DataFrame({"text": [t[:40] for t in texts], "n_tokens": features["attention_mask"].sum(dim=1).tolist()})
    .assign(truncated=lambda d: d["n_tokens"] >= model.max_seq_length)
)
display(df_tokens)

,text,n_tokens,truncated
0,Go to Settings > Security > Reset Passwo,28,False
1,We offer full refunds within 30 days of,28,False
2,Go to Account Settings > Subscription >,24,False
3,"We accept Visa, Mastercard, American Exp",23,False
4,Go to Settings > Security > Reset Passwo,543,False


Embeddings are normalized, so a dot product is cosine similarity:

In [11]:
n         = 6                                                                   # top-left slice; the full 20x20 matrix is too wide to read
sim_names = df_faq["question"].str[:28].tolist()[:n]
pd.DataFrame(vectors[:n] @ vectors[:n].T, index=sim_names, columns=range(n)).round(2)

,0,1,2,3,4,5
How do I reset my password?,1.00,0.67,0.79,0.66,0.80,0.61
What's your refund policy?,0.67,1.01,0.78,0.79,0.67,0.72
How do I cancel my subscript,0.79,0.78,1.00,0.70,0.73,0.64
What payment methods do you,0.66,0.79,0.70,1.01,0.69,0.65
How do I update my profile i,0.80,0.67,0.73,0.69,1.00,0.64
How long does shipping take?,0.61,0.72,0.64,0.65,0.64,1.00


To retrieve rather than compare, encode the question with `Objective.QUERY` and score it against the `DOCUMENT` vectors from above. On a model that defines prompts the query and document sides are encoded differently; on `all-MiniLM-L6-v2` all three objectives give identical vectors, which the last cell checks:

In [12]:
query     = "How can I get my money back?"
query_vec = encode_texts(model, [query], Objective.QUERY, **enc)[0]
df_ranked = (
    df_faq[["question", "category"]]
    .assign(score=doc_vecs @ query_vec)
    .sort_values("score", ascending=False)
    .head(5)
    .reset_index(drop=True)
)
display(df_ranked.round(3))

,question,category,score
0,What's your refund policy?,Billing,0.834
1,How do I initiate a return?,Returns,0.808
2,What is your return policy?,Returns,0.807
3,How do I cancel my subscription?,Billing,0.767
4,How do I delete my account?,Account,0.744


In [13]:
plain_vec = encode_texts(model, [query], **enc)[0]    # SYMMETRIC: no query prompt
print(np.allclose(query_vec, plain_vec), round(float(query_vec @ plain_vec), 3))

False 0.887


## Merging models

`SentenceTransformer.merge` combines compatible fine-tuned checkpoints into one model without another training run, using [mergekit](https://github.com/arcee-ai/mergekit) (`linear`, `slerp`, `ties`, `dare`, and more). It needs `sentence-transformers[merge]>=6.2`, which the workspace now requires. The inputs must share an architecture, module configuration, and tokenizer; config and prompts come from the first model. Merging runs in FP32 on CPU and holds the inputs and result in memory, so it is separate from the `model` loaded above.

Here a `linear` merge takes a 0.6 / 0.4 weighted average of two MiniLM-L6 models:

In [14]:
MERGE_MODELS: List[str] = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/multi-qa-MiniLM-L6-cos-v1",
]
hf_cache: Dict[str, Any] = dict(cache_folder=read_cache_dir(app="huggingface/hub"))
merged = SentenceTransformer.merge(MERGE_MODELS, method="linear", parameters={"weight": [0.6, 0.4]}, **hf_cache)    # weights per model, in order

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encode the FAQ questions with the merged model and compare each vector to the same question from both source models, averaged per category. A linear merge should land close to both, nearer the one with the larger weight. Use `method="slerp"` with `parameters={"t": 0.5}` to interpolate between two models instead, and `merged.save_pretrained(path)` to keep the result:

In [15]:
merged_vecs  = merged.encode(questions, **enc)
source_vecs  = [SentenceTransformer(name, **hf_cache).encode(questions, **enc) for name in MERGE_MODELS]
df_merge_sim = (
    pd.DataFrame({
        "category": df_faq["category"],
        **{name.split("/")[-1]: (merged_vecs * vecs).sum(axis=1) for name, vecs in zip(MERGE_MODELS, source_vecs)},    # row-wise cosine (vectors are normalized)
    })
    .groupby("category")
    .mean()
)
display(df_merge_sim.round(3))
print(merged_vecs.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,all-MiniLM-L6-v2,multi-qa-MiniLM-L6-cos-v1
category,,
Account,0.947,0.909
Billing,0.955,0.901
Orders,0.960,0.913
Product,0.953,0.902
Returns,0.947,0.885
Security,0.942,0.899
Shipping,0.952,0.911
Technical,0.953,0.899


(20, 384)


## Visualizing embeddings with UMAP

`UMAPProjection` reduces the FAQ question vectors to 2-D so they can be plotted. It defaults to `ProjectionMode.VISUALIZE` (2 components, cosine metric), and `to_frame` returns the coordinates as `umap_0`, `umap_1` columns, with the labels attached for coloring, here each question's `category`. With only 20 rows (2-4 per category), expect loose clusters; UMAP clamps `n_neighbors` to the sample count.

The black stars are queries, encoded with `Objective.QUERY` and placed in the already-fitted space with `transform`, so they don't move the other points (the placement of a new point is approximate). The legend lists them under their own "Queries" heading, apart from the categories. Edit or extend `viz_queries` to see where other questions land.

In [16]:
def plot_scatter(
    df_plot:      pd.DataFrame,
    x:            str,
    y:            str,
    color:        Optional[str] = None,
    hover_cols:   Optional[List[str]] = None,
    title:        str = "",
    legend_title: Optional[str] = None,
    **kwargs:     Any,
) -> go.Figure:
    """Scatter two DataFrame columns with Plotly Express; independent of how the coordinates were made.

    Args:
        df_plot (pd.DataFrame): Frame holding the coordinate columns (plus any color or hover columns).
        x (str): Column for the horizontal axis.
        y (str): Column for the vertical axis.
        color (Optional[str]): Column to color series by; omit for a single series with no legend.
        hover_cols (Optional[List[str]]): Extra columns to show on hover.
        title (str): Plot title.
        legend_title (Optional[str]): Heading of the color legend, pinned top-right so `highlight_points` can add boxes below it; defaults to `color`.
        **kwargs (Any): Forwarded to `px.scatter` (e.g. `facet_col`, `symbol`, `opacity`); size defaults to 700x500.

    Returns:
        go.Figure: The figure; call `.show()` to render or `.update_layout()` to adjust.

    Examples:
        >>> df_scores = df_experiments.assign(outcome=lambda d: d["accuracy"].gt(0.9).map({True: "pass", False: "fail"}))
        >>> fig = plot_scatter(df_scores, x="latency_ms", y="accuracy", color="outcome", hover_cols=["run_id"], title="Eval runs")
    """
    fig = px.scatter(df_plot, x=x, y=y, color=color, hover_data=hover_cols, title=title, **{"width": 700, "height": 500, **kwargs})
    return fig if not color else fig.update_layout(legend=dict(title_text=legend_title or color, x=1.02, xanchor="left", y=1, yanchor="top"))

In [17]:
def highlight_points(
    fig:       go.Figure,
    df_points: pd.DataFrame,
    x:         str,
    y:         str,
    name_col:  str,
    hover_col: Optional[str] = None,
    group:     str = "Highlights",
    **kwargs:  Any,
) -> go.Figure:
    """Overlay emphasized points on an existing figure, keyed in their own box beside the plot, apart from any color legend.

    Args:
        fig (go.Figure): Figure to draw on, e.g. from `plot_scatter`.
        df_points (pd.DataFrame): One row per point to emphasize; must be on the same scale as `fig`.
        x (str): Column for the horizontal axis.
        y (str): Column for the vertical axis.
        name_col (str): Short label for each point, drawn next to it and listed in the key (keep it brief, e.g. "Q1").
        hover_col (Optional[str]): Longer text shown on hover; defaults to `name_col`.
        group (str): Heading of the key box.
        **kwargs (Any): Override the marker settings (default: black star, size 18, white outline).

    Returns:
        go.Figure: `fig`, with one trace per row and the key box added bottom-right.

    Examples:
        >>> fig = plot_scatter(df_runs, x="latency_ms", y="accuracy", color="model", legend_title="Model")
        >>> df_best = df_runs.nlargest(2, "accuracy").assign(rank=["#1", "#2"])
        >>> fig = highlight_points(fig, df_best, x="latency_ms", y="accuracy", name_col="rank", hover_col="run_id", group="Best runs")
    """
    marker = {"symbol": "star", "size": 18, "color": "black", "line": {"width": 2, "color": "white"}, **kwargs}
    hovers = df_points[hover_col if hover_col else name_col]
    for px_, py_, name, hover in zip(df_points[x], df_points[y], df_points[name_col], hovers):
        fig.add_trace(go.Scatter(
            x=[px_], y=[py_], mode="markers+text", name=str(name), text=[str(name)], textposition="top center", marker=marker,
            showlegend=False, hovertext=str(hover),
        ))
    # An annotation, not a second legend: it renders the same in every viewer and cannot merge into the color legend.
    key = "<br>".join([f"<b>{group}</b>"] + [f"★ {name}" for name in df_points[name_col]])
    return fig.add_annotation(
        text=key, xref="paper", yref="paper", x=1.02, y=0, xanchor="left", yanchor="bottom", align="left", showarrow=False,
    ).update_layout(margin=dict(r=120))

In [18]:
viz_queries = ["How can I get my money back?"]                               # add or edit queries to see where each lands
viz_qvecs   = encode_texts(model, viz_queries, Objective.QUERY, **enc)
projection  = UMAPProjection()                                               # VISUALIZE: 2 components, cosine metric
df_proj     = projection.to_frame(doc_vecs, labels=df_faq["category"]).assign(question=df_faq["question"].tolist())
df_query    = (
    pd.DataFrame(projection.transform(viz_qvecs), columns=["umap_0", "umap_1"])    # projected into the fitted space; doesn't move the other points
    .assign(question=viz_queries, tag=[f"Q{i}" for i in range(1, len(viz_queries) + 1)])
)
fig = plot_scatter(df_proj, x="umap_0", y="umap_1", color="label", hover_cols=["question"], title="FAQ questions by category (UMAP)", legend_title="Category")
fig = highlight_points(fig, df_query, x="umap_0", y="umap_1", name_col="tag", hover_col="question", group="Queries")
fig.show()
display(df_query[["tag", "question"]])                                       # full query text lives here and on hover, not in the legend
print(projection.get_parameters())

/Users/akamlani/Library/CloudStorage/Dropbox/dev-platform/repositories/projects/velari-suite/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


,tag,question
0,Q1,How can I get my money back?


{'mode': 'visualize', 'n_components': 2, 'n_neighbors': 15}


For downstream use rather than plotting, fit a `ProjectionMode.TRANSFORM` projection (10 components by default; 5 here) on 80% of the FAQ rows and apply `transform` to the held-out 20%. They land in the same reduced space as the fitted vectors, so they can feed clustering or a classifier:

In [19]:
df_fit      = df_faq.sample(frac=0.8, random_state=42)
df_holdout  = df_faq.drop(df_fit.index)
reducer     = UMAPProjection(mode=ProjectionMode.TRANSFORM, n_components=5)
reduced     = reducer.fit_transform(vectors[df_fit.index])
new_reduced = reducer.transform(encode_texts(model, df_holdout["question"].tolist(), **enc))
print(reduced.shape, new_reduced.shape)

/Users/akamlani/Library/CloudStorage/Dropbox/dev-platform/repositories/projects/velari-suite/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


(16, 5) (4, 5)


## Clustering with HDBSCAN

Do the FAQ embeddings group into the known categories on their own? `HDBSCANDriver` finds the number of clusters itself and labels outliers as noise (`-1`); `KMeansDriver` with one cluster per category is the known-k baseline.

Cluster the reduced vectors, not the 2-D plot coordinates: the 2-D picture distorts distances, and raw high-dimensional distances are too flat for density clustering. A `TRANSFORM` projection to 5 components is fitted on all 20 rows, unlike the holdout cell above. `min_cluster_size=2` and `min_samples=1` suit a corpus this small; larger values label almost everything noise.

In [20]:
reducer_5d  = UMAPProjection(mode=ProjectionMode.TRANSFORM, n_components=5)
reduced_all = reducer_5d.fit_transform(doc_vecs)                          # all 20 rows, cluster in this space
hdbscan_drv = HDBSCANDriver(min_cluster_size=2, min_samples=1)
kmeans_drv  = KMeansDriver(n_clusters=df_faq["category"].nunique())       # known-k baseline: one cluster per category
df_hdbscan  = hdbscan_drv.to_frame(reduced_all, labels=df_faq["category"])    # cluster (-1 = noise), label, probability, outlier_score
df_kmeans   = kmeans_drv.to_frame(reduced_all, labels=df_faq["category"])
print(hdbscan_drv.get_parameters())
print(kmeans_drv.get_parameters())

{'min_cluster_size': 2, 'num_clusters': 6, 'noise_cnt': 1, 'probabilities_mean': 0.919362730245305}
{'n_clusters': 8, 'inertia': 4.065215587615967, 'n_iter': 2}


/Users/akamlani/Library/CloudStorage/Dropbox/dev-platform/repositories/projects/velari-suite/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Compare the clusters to the known categories. The cross-table shows which categories each HDBSCAN cluster mixes (row `-1` is noise). `scorer_clustering` rates the agreement in five groups, shown as the first index level; every score is 1.0 for a perfect match except `noise_fraction`:

- **`pairwise`:** `adjusted_rand` is chance-corrected (near 0 is random); `fowlkes_mallows` is not.
- **`information`:** `adjusted_mutual_info` is chance-corrected, `normalized_mutual_info` is not and rises as the number of clusters grows.
- **`entropy`:** `homogeneity` is 1 when every cluster holds one category, `completeness` when every category sits in one cluster. Read them together, because many tiny clusters score a perfect homogeneity; `v_measure` is their harmonic mean.
- **`matching`:** `purity` is the share of rows in their cluster's majority category, `matched_accuracy` the share correct if each cluster is named after its best category.
- **`outliers`:** `noise_fraction` is the share of rows HDBSCAN left unclustered (0 for KMeans).

In [21]:
def flat_scores(scores: Dict[str, Dict[str, float]]) -> Dict[Any, float]:
    """Flatten grouped scores to `{(group, metric): value}`, so a DataFrame shows the groups as an index level.

    Args:
        scores (Dict[str, Dict[str, float]]): Output of `scorer_clustering`.

    Returns:
        Dict[Any, float]: One entry per metric, keyed by its `(group, metric)` pair.

    Examples:
        >>> flat_scores(scorer_clustering(df_faq["category"], df_hdbscan["cluster"]))[("pairwise", "adjusted_rand")]
    """
    return {(group, metric): value for group, metrics in scores.items() for metric, value in metrics.items()}


display(pd.crosstab(df_hdbscan["cluster"], df_hdbscan["label"]))
df_scores = pd.DataFrame({
    "hdbscan": flat_scores(scorer_clustering(df_faq["category"], df_hdbscan["cluster"])),     # noise (-1) counts as one more cluster
    "kmeans":  flat_scores(scorer_clustering(df_faq["category"], df_kmeans["cluster"])),
})
display(df_scores.round(3))

label,Account,Billing,Orders,Product,Returns,Security,Shipping,Technical
cluster,,,,,,,,
-1,0,0,0,1,0,0,0,0
0,3,1,0,0,0,1,0,2
1,0,1,0,0,0,1,0,0
2,0,0,1,0,0,0,1,0
3,0,2,0,0,0,0,0,0
4,0,0,1,0,1,0,0,0
5,0,0,0,1,1,0,2,0


hdbscan  kmeans
pairwise    adjusted_rand             0.152   0.225
            fowlkes_mallows           0.261   0.294
information adjusted_mutual_info      0.208   0.235
            normalized_mutual_info    0.620   0.682
entropy     homogeneity               0.578   0.682
            completeness              0.670   0.682
            v_measure                 0.620   0.682
matching    purity                    0.550   0.600
            matched_accuracy          0.550   0.600
outliers    noise_fraction            0.050   0.000

Assign the queries from the plot section to the fitted clusters. They are projected with the same `reducer_5d`, then `predict` gives a hard label (`-1` if the query sits outside every cluster) and `predict_proba` a soft membership in each cluster. The two can disagree: a query can be noise by `predict` yet still lean toward one cluster by membership.

In [22]:
q_reduced = reducer_5d.transform(viz_qvecs)                                # same space the clusters were fitted in
q_proba   = hdbscan_drv.predict_proba(q_reduced)                           # (n_queries, n_clusters); zero columns if no clusters were found
has_proba = q_proba.shape[1] > 0
df_assign = pd.DataFrame({
    "tag":         df_query["tag"],
    "question":    viz_queries,
    "cluster":     hdbscan_drv.predict(q_reduced),                         # hard label, -1 = noise
    "top_cluster": q_proba.argmax(axis=1) if has_proba else np.full(len(viz_queries), -1),
    "membership":  q_proba.max(axis=1)    if has_proba else np.zeros(len(viz_queries)),
})
display(df_assign.round(3))

,tag,question,cluster,top_cluster,membership
0,Q1,How can I get my money back?,-1,5,0.719


The table above keeps only each query's best cluster. To see how close the others are, score and rank every cluster for every query. `rank_clusters` uses only `predict` and `predict_proba`, which every clustering driver implements, so the same call works for `KMeansDriver` or any future model.

- `membership` is `predict_proba`'s soft score for that row's cluster: a closeness measure, not a calibrated probability, and the scores need not sum to 1. It has no density cutoff, so a query that `predict` calls noise can still lean toward one cluster.
- `assigned` is True on the row whose cluster `predict` chose for the query (the `cluster` column of the table above). For HDBSCAN no row is marked when the query is noise: `approximate_predict` finds the query's nearest training points, turns their mutual-reachability distance into a density level (lambda = 1 / distance), walks up the cluster tree to the cluster that exists at that density, and returns `-1` unless that cluster is one of the selected ones. KMeans has no noise, so exactly one row per query is marked.
- `rank` 1 is the highest membership for that query; exact ties rank in cluster order.
- `majority_category` is the most common FAQ category inside that cluster, since the cluster ids are arbitrary.

In [ ]:
def cluster_name(label: int) -> str:
    """Readable cluster name for a legend: `noise` for HDBSCAN's `-1`, else `C<label>`."""
    return "noise" if label == -1 else f"C{label}"


def rank_clusters(driver: Any, X: np.ndarray, tags: Any, df_clusters: pd.DataFrame) -> pd.DataFrame:
    """Score and rank every cluster for every query; uses only the shared clustering interface (`predict`, `predict_proba`)."""
    proba      = driver.predict_proba(X)                                                   # (n_queries, n_clusters); no columns if none were found
    prediction = dict(zip(tags, driver.predict(X)))                                        # tag -> cluster chosen by predict (-1 = noise for HDBSCAN)
    majority   = pd.crosstab(df_clusters["cluster"], df_clusters["label"]).idxmax(axis=1)  # cluster -> most common category
    return (
        pd.DataFrame(proba, columns=list(range(proba.shape[1]))).assign(tag=list(tags))
        .melt(id_vars="tag", var_name="cluster_id", value_name="membership")
        .assign(
            cluster           = lambda d: d["cluster_id"].map(cluster_name),
            assigned          = lambda d: d["cluster_id"].eq(d["tag"].map(prediction)),
            rank              = lambda d: d.groupby("tag")["membership"].rank(ascending=False, method="first").astype(int),
            majority_category = lambda d: d["cluster_id"].map(majority),
        )
        .sort_values(["tag", "rank"])
        .loc[:, ["tag", "cluster", "membership", "assigned", "rank", "majority_category"]]
        .reset_index(drop=True)
    )


df_ranking = rank_clusters(hdbscan_drv, q_reduced, df_query["tag"], df_hdbscan)
display(df_ranking.round(4))                                                              # far clusters score near 0, so 3 decimals would hide the order

Draw the clusters on the 2-D UMAP from the plot section (run that cell first). Color is the HDBSCAN cluster; hover a point to see its true category, and compare with the cross-table above. The stars are the queries, with their assigned cluster on hover.

In [24]:
df_clustered = df_proj.assign(
    cluster     = df_hdbscan["cluster"].map(cluster_name),         # strings, so Plotly colors them as categories
    probability = df_hdbscan["probability"].round(2),
)
df_query_hover = df_query.assign(hover=[f"{q} (cluster: {cluster_name(c)})" for q, c in zip(df_assign["question"], df_assign["cluster"])])
fig = plot_scatter(
    df_clustered, x="umap_0", y="umap_1", color="cluster",
    hover_cols=["question", "label", "probability"], title="FAQ questions by HDBSCAN cluster (UMAP)", legend_title="Cluster",
)
fig = highlight_points(fig, df_query_hover, x="umap_0", y="umap_1", name_col="tag", hover_col="hover", group="Queries")
fig.show()

How to read this: with 20 questions and 2-4 per category, agreement is evidence about the embeddings, not a verdict, and it moves with the model. Noise points are a result, not an error: HDBSCAN declines to force an outlier into a cluster. `predict` is an approximation for new points, and `predict_proba` is a membership score, not a calibrated probability.